# Notebook 6: Hyperparameter Tuning
## CrossValidator + ParamGridBuilder for Optimal Model Selection

Uses PySpark's `CrossValidator` and `ParamGridBuilder` to systematically
search for the best hyperparameters. Includes Spark UI analysis.

**Important**: Take **Spark UI screenshots** at `http://localhost:4040` during execution!

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, GBTClassifier
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

sns.set_theme(style='whitegrid')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Spark Session
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_Tuning')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', 200)
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} | UI: http://localhost:4040')
print('\n⚠ IMPORTANT: Open http://localhost:4040 in your browser to take Spark UI screenshots!')

In [ ]:
# ============================================================================
# Cell 3: Load Data
# ============================================================================
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

train_df = spark.read.parquet(os.path.join(PROCESSED_DIR, 'train.parquet'))
test_df = spark.read.parquet(os.path.join(PROCESSED_DIR, 'test.parquet'))

train_df = train_df.withColumnRenamed('is_viral', 'label').cache()
test_df = test_df.withColumnRenamed('is_viral', 'label').cache()

print(f'Training: {train_df.count():,} | Test: {test_df.count():,}')

In [ ]:
# ============================================================================
# Cell 4: Define Evaluator
# ============================================================================
evaluator = BinaryClassificationEvaluator(
    labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)
print('Evaluator: BinaryClassificationEvaluator (AUC-ROC)')

## Tuning 1: Logistic Regression

**Parameters to tune**:
- `regParam`: Regularization strength (prevents overfitting)
- `elasticNetParam`: L1/L2 mix (0=L2, 1=L1)
- `maxIter`: Maximum iterations for convergence

In [ ]:
# ============================================================================
# Cell 5: Logistic Regression Hyperparameter Tuning
# ============================================================================
print('TUNING: Logistic Regression')
print('=' * 60)

lr = LogisticRegression(featuresCol='features', labelCol='label')

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .addGrid(lr.maxIter, [50, 100])
    .build()
)

print(f'Parameter grid size: {len(lr_param_grid)} combinations')
print(f'With 3-fold CV: {len(lr_param_grid) * 3} total model fits')
print('\nGrid:')
print('  regParam: [0.001, 0.01, 0.1]')
print('  elasticNetParam: [0.0, 0.5, 1.0]')
print('  maxIter: [50, 100]')

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=42
)

start_time = time.time()
print('\n>>> Take a Spark UI screenshot now! (http://localhost:4040/jobs)')
lr_cv_model = lr_cv.fit(train_df)
lr_cv_time = time.time() - start_time

print(f'\nCross-validation completed in {lr_cv_time:.1f}s')

# Best model parameters
best_lr = lr_cv_model.bestModel
print(f'Best regParam: {best_lr._java_obj.getRegParam()}')
print(f'Best elasticNetParam: {best_lr._java_obj.getElasticNetParam()}')
print(f'Best maxIter: {best_lr._java_obj.getMaxIter()}')

# CV scores
lr_cv_scores = lr_cv_model.avgMetrics
print(f'\nCV scores (AUC-ROC): {[round(s, 4) for s in lr_cv_scores]}')
print(f'Best CV AUC-ROC: {max(lr_cv_scores):.4f}')

## Tuning 2: Random Forest

**Parameters to tune**:
- `numTrees`: Number of trees in the forest
- `maxDepth`: Maximum depth per tree
- `minInstancesPerNode`: Minimum samples for splitting

In [ ]:
# ============================================================================
# Cell 6: Random Forest Hyperparameter Tuning
# ============================================================================
print('TUNING: Random Forest')
print('=' * 60)

rf = RandomForestClassifier(featuresCol='features', labelCol='label', seed=42)

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .addGrid(rf.minInstancesPerNode, [1, 5])
    .build()
)

print(f'Parameter grid size: {len(rf_param_grid)} combinations')
print(f'With 3-fold CV: {len(rf_param_grid) * 3} total model fits')

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,  # Lower parallelism for memory-heavy RF
    seed=42
)

start_time = time.time()
print('\n>>> Take a Spark UI screenshot now! (http://localhost:4040/stages)')
rf_cv_model = rf_cv.fit(train_df)
rf_cv_time = time.time() - start_time

print(f'\nCross-validation completed in {rf_cv_time:.1f}s')

best_rf = rf_cv_model.bestModel
print(f'Best numTrees: {best_rf.getNumTrees}')
print(f'Best maxDepth: {best_rf._java_obj.getMaxDepth()}')

rf_cv_scores = rf_cv_model.avgMetrics
print(f'Best CV AUC-ROC: {max(rf_cv_scores):.4f}')

## Tuning 3: Gradient Boosted Trees

**Parameters to tune**:
- `maxIter`: Number of boosting rounds
- `maxDepth`: Depth per tree
- `stepSize`: Learning rate

In [ ]:
# ============================================================================
# Cell 7: GBT Hyperparameter Tuning
# ============================================================================
print('TUNING: Gradient Boosted Trees')
print('=' * 60)

gbt = GBTClassifier(featuresCol='features', labelCol='label', seed=42)

gbt_param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [30, 50, 100])
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.stepSize, [0.05, 0.1, 0.2])
    .build()
)

print(f'Parameter grid size: {len(gbt_param_grid)} combinations')
print(f'With 3-fold CV: {len(gbt_param_grid) * 3} total model fits')

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)

start_time = time.time()
print('\n>>> Take a Spark UI screenshot now! (http://localhost:4040/storage)')
gbt_cv_model = gbt_cv.fit(train_df)
gbt_cv_time = time.time() - start_time

print(f'\nCross-validation completed in {gbt_cv_time:.1f}s')

best_gbt = gbt_cv_model.bestModel
print(f'Best maxIter: {best_gbt._java_obj.getMaxIter()}')
print(f'Best maxDepth: {best_gbt._java_obj.getMaxDepth()}')
print(f'Best stepSize: {best_gbt._java_obj.getStepSize()}')

gbt_cv_scores = gbt_cv_model.avgMetrics
print(f'Best CV AUC-ROC: {max(gbt_cv_scores):.4f}')

In [ ]:
# ============================================================================
# Cell 8: Tuning Summary & Best Models on Test Set
# ============================================================================
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='f1')

tuning_results = []

for name, cv_model, cv_time in [
    ('Logistic Regression (Tuned)', lr_cv_model, lr_cv_time),
    ('Random Forest (Tuned)', rf_cv_model, rf_cv_time),
    ('GBT (Tuned)', gbt_cv_model, gbt_cv_time)
]:
    preds = cv_model.transform(test_df)
    auc = evaluator.evaluate(preds)
    acc = acc_eval.evaluate(preds)
    f1 = f1_eval.evaluate(preds)
    
    tuning_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1-Score': round(f1, 4),
        'AUC-ROC': round(auc, 4),
        'CV Time (s)': round(cv_time, 1),
        'Best CV Score': round(max(cv_model.avgMetrics), 4)
    })

tuning_df = pd.DataFrame(tuning_results)

print('=' * 80)
print('HYPERPARAMETER TUNING RESULTS (Test Set Performance)')
print('=' * 80)
print(tuning_df.to_string(index=False))

best_idx = tuning_df['AUC-ROC'].idxmax()
print(f'\n🏆 Best tuned model: {tuning_df.loc[best_idx, "Model"]}')

In [ ]:
# ============================================================================
# Cell 9: Visualization
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Performance
plot_data = tuning_df[['Model', 'Accuracy', 'F1-Score', 'AUC-ROC']].melt(
    id_vars='Model', var_name='Metric', value_name='Score'
)
sns.barplot(data=plot_data, x='Model', y='Score', hue='Metric', ax=axes[0], palette='viridis')
axes[0].set_title('Tuned Model Performance', fontsize=14)
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)

# CV Time
sns.barplot(data=tuning_df, x='Model', y='CV Time (s)', ax=axes[1], palette='magma')
axes[1].set_title('Cross-Validation Time', fontsize=14)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'hyperparameter_tuning.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save for Tableau
tuning_df.to_csv(
    os.path.join(PROJECT_ROOT, 'tableau', 'data', 'tuning_results.csv'),
    index=False
)
print('Results saved to tableau/data/tuning_results.csv')

In [ ]:
# ============================================================================
# Cell 10: Save Best Models
# ============================================================================
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')

lr_cv_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, 'best_lr_tuned'))
rf_cv_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, 'best_rf_tuned'))
gbt_cv_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, 'best_gbt_tuned'))

print('Best tuned models saved.')
print('\nREMINDER: Save Spark UI screenshots from http://localhost:4040')
print('  - Jobs tab screenshot')
print('  - Stages tab screenshot')
print('  - Storage tab screenshot')
print('  - Executors tab screenshot')

spark.stop()
print('\nSpark session stopped.')